# CN 数据质量专项检查

检查 CSV 源数据完整性、Label 质量、Qlib 特征可用性。按顺序执行 Cell。

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CSV_DIR = ROOT / "data" / "csv_source"
print(f"CSV directory: {CSV_DIR}")
print(f"Files: {len(list(CSV_DIR.glob('*.csv')))} files")

## 1. 全部 CSV 数据扫描

In [ ]:
results = []
for f in sorted(CSV_DIR.glob("*.csv")):
    sym = f.stem
    try:
        df = pd.read_csv(f, parse_dates=["date"]).sort_values("date")
        n = len(df)
        df["ret"] = df["close"].shift(-10) / df["close"] - 1
        valid_label = df["ret"].notna() & (df["ret"].abs() > 1e-10)
        ohlc_ok = bool((df["high"] >= df["open"]).fillna(True).all() and (df["low"] <= df["open"]).fillna(True).all())
        results.append({
            "symbol": sym,
            "market": "CN" if sym.isdigit() else "US",
            "rows": n, "start": str(df["date"].min())[:10], "end": str(df["date"].max())[:10],
            "ohlc_ok": ohlc_ok, "label_pct": valid_label.sum() / max(n,1) * 100,
        })
    except Exception as e:
        results.append({"symbol": sym, "error": str(e)})

df_r = pd.DataFrame(results)
cn = df_r[df_r["market"] == "CN"].sort_values("label_pct", ascending=False)
print(f"CN symbols: {len(cn)}  |  Total rows: {cn["rows"].sum():,}")
print(f"OHLC valid: {cn["ohlc_ok"].sum()}/{len(cn)}")
print(f"Avg label quality: {cn["label_pct"].mean():.1f}%")

## 2. Label 质量排名

In [ ]:
print("Top 20 (best labels):")
for _, r in cn.head(20).iterrows():
    print(f"  {r['symbol']:8s}  {r['rows']:5d} rows  {r['start']}->{r['end']}  label={r['label_pct']:.1f}%")
print()
print("Bottom 20 (worst labels):")
for _, r in cn.tail(20).iterrows():
    print(f"  {r['symbol']:8s}  {r['rows']:5d} rows  {r['start']}->{r['end']}  label={r['label_pct']:.1f}%")

In [ ]:
# Label quality distribution
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(cn["label_pct"], bins=50, color="#6366f1", edgecolor="white", alpha=0.8)
ax.axvline(99.0, color="green", ls="--", label="99% threshold")
ax.axvline(95.0, color="orange", ls="--", label="95% threshold")
ax.set_title("CN Label Quality Distribution", fontweight="bold")
ax.set_xlabel("Valid Label %"); ax.legend()
plt.tight_layout(); plt.show()

## 3. 数据缺失诊断

In [ ]:
# Diagnose worst symbol
worst = cn.iloc[-1]
print(f"Worst symbol: {worst['symbol']} (label={worst['label_pct']:.1f}%)")
df = pd.read_csv(CSV_DIR / f"{worst['symbol']}.csv", parse_dates=["date"]).sort_values("date")
df["gap"] = df["date"].diff().dt.days
gaps = df[df["gap"] > 5]
print(f"Date gaps > 5 days: {len(gaps)}")
if len(gaps) > 0:
    print(gaps[["date","gap"]].head(10).to_string())
df["ret"] = df["close"].shift(-10) / df["close"] - 1
print(f"Label NaN: {df['ret'].isna().sum()}  Zero: {(df['ret'].abs() < 1e-10).sum()}")

## 4. OHLC 异常检测

In [ ]:
# Check OHLC validity
bad = cn[cn["ohlc_ok"] == False]
print(f"OHLC anomalies: {len(bad)} symbols")
for _, r in bad.iterrows():
    df = pd.read_csv(CSV_DIR / f"{r['symbol']}.csv", parse_dates=["date"])
    hi_lo = (df["high"] < df["open"]).sum()
    lo_hi = (df["low"] > df["open"]).sum()
    print(f"  {r['symbol']}: high<open={hi_lo}  low>open={lo_hi}")

## 5. Qlib 数据可用性

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from qlib.data import D

safe_qlib_init(build_qlib_init_cfg(None, market="cn"))
test_syms = cn.head(10)["symbol"].tolist()
print(f"Checking Qlib data for top 10...")
for sym in test_syms:
    try:
        df = D.features([sym], ["$close"], start_time="2021-01-01", end_time="2026-06-25")
        print(f"  {sym}: {len(df):4d} rows  last={str(df.index.get_level_values('datetime').max())[:10]}")
    except Exception as e:
        print(f"  {sym}: ERROR {e}")

# Instruments file check
instr_path = ROOT / "data" / "watchlist" / "instruments" / "cn.txt"
if instr_path.exists():
    instr_syms = [l.split(chr(9))[0] for l in instr_path.read_text().splitlines() if l.strip()]
    missing = [s for s in instr_syms if not (CSV_DIR / f"{s}.csv").exists()]
    print(f"
Instruments file: {len(instr_syms)} symbols, {len(missing)} missing CSV")

## 6. 摘要 & 建议

In [ ]:
print("=" * 60)
print("DATA QUALITY AUDIT — SUMMARY")
print("=" * 60)
print(f"CN symbols: {len(cn)}")
print(f"Avg label quality: {cn["label_pct"].mean():.1f}%")
print(f">=99% clean: {(cn["label_pct"] >= 99).sum()}")
print(f">=95% clean: {(cn["label_pct"] >= 95).sum()}")
print(f"OHLC valid: {cn["ohlc_ok"].sum()}/{len(cn)}")
print()
print("Training-ready (99%+):")
best = cn[cn["label_pct"] >= 99]["symbol"].tolist()
print(f"  {len(best)} symbols")
print()
report = {
    "generated_at": datetime.now().isoformat(),
    "total": int(len(cn)),
    "avg_quality": float(cn["label_pct"].mean()),
    "n99": int((cn["label_pct"] >= 99).sum()),
    "ohlc_ok": int(cn["ohlc_ok"].sum()),
}
with open(ROOT / "artifacts" / "dashboard" / "data_quality_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("Report saved.")